# Stage 1 — Mine positive và hard negatives

Notebook này dùng **short index có BM25+dense+mapping** và **long index chỉ cần dense**. Chỉ dùng `seed_2026/train.json`; không mine từ validation/test. Hãy bật Kaggle Accelerator `GPU T4 x2` và Internet cho lần đầu tải model.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/w4ngg/DSC-Legal-IR-QA.git"
REPO_ROOT = Path("/kaggle/working/DSC-Legal-IR-QA")
RETRIEVAL_ROOT = REPO_ROOT / "retrieval"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    print("Repository đã tồn tại, tái sử dụng:", REPO_ROOT)

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
assert (RETRIEVAL_ROOT / "src/legal_ir/mine_reranker_stage1.py").is_file(), (
    "GitHub main chưa chứa code Stage 1. Hãy commit/push source mới trước."
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(RETRIEVAL_ROOT)],
    check=True,
)
print("Commit:", SOURCE_COMMIT)


In [ ]:
import torch

subprocess.run(["nvidia-smi", "-L"], check=True)
assert torch.cuda.is_available(), "CUDA không khả dụng"
assert torch.cuda.device_count() == 2, (
    f"Notebook mining được thiết kế cho T4x2, hiện thấy {torch.cuda.device_count()} GPU"
)
for gpu_id in range(torch.cuda.device_count()):
    print(gpu_id, torch.cuda.get_device_name(gpu_id))

HF_HOME = Path("/kaggle/working/hf-cache")
MULTI_GPU_TMP = Path("/kaggle/working/reranker-multi-gpu-tmp")
HF_HOME.mkdir(parents=True, exist_ok=True)
MULTI_GPU_TMP.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["LEGAL_IR_RERANKER_MULTI_GPU_TMPDIR"] = str(MULTI_GPU_TMP)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Disk trống:", f"{shutil.disk_usage('/kaggle/working').free / 1024**3:.2f} GiB")


## Tạo train/validation/test split

Cell này tạo split deterministic từ `IR/train.json`: 5.600 train, 700 validation và 700 test với seed 2026. Câu hỏi trùng sau normalize được giữ trong cùng một split để tránh leakage. Không dùng `--overwrite` trừ khi bạn chủ động muốn thay toàn bộ bốn artifact split.

In [ ]:
SOURCE_TRAIN_JSON = REPO_ROOT / "IR/train.json"
GENERATED_SPLIT_DIR = Path("/kaggle/working/artifacts/splits/seed_2026")
BUILD_SPLIT_FROM_REPO = True
OVERWRITE_SPLIT = False

if BUILD_SPLIT_FROM_REPO:
    assert SOURCE_TRAIN_JSON.is_file(), SOURCE_TRAIN_JSON
    split_files = [
        GENERATED_SPLIT_DIR / "train.json",
        GENERATED_SPLIT_DIR / "val.json",
        GENERATED_SPLIT_DIR / "test.json",
        GENERATED_SPLIT_DIR / "split_manifest.json",
    ]
    existing = [path for path in split_files if path.exists()]
    if len(existing) == len(split_files) and not OVERWRITE_SPLIT:
        print("Split đầy đủ đã tồn tại, tái sử dụng:", GENERATED_SPLIT_DIR)
    elif existing and not OVERWRITE_SPLIT:
        raise RuntimeError(
            "Split đang dở dang. Xóa đúng thư mục output hoặc đặt OVERWRITE_SPLIT=True: "
            + ", ".join(str(path) for path in existing)
        )
    else:
        command = [
            sys.executable,
            "-m",
            "legal_ir.create_val_test",
            "--input", str(SOURCE_TRAIN_JSON),
            "--output-dir", str(GENERATED_SPLIT_DIR),
            "--val-size", "700",
            "--test-size", "700",
            "--seed", "2026",
        ]
        if OVERWRITE_SPLIT:
            command.append("--overwrite")
        subprocess.run(command, check=True)
    SPLIT_DIR = GENERATED_SPLIT_DIR
else:
    # Nếu split đã được upload thành Kaggle Dataset, sửa đường dẫn này.
    SPLIT_DIR = Path("/kaggle/input/<split-dataset>/seed_2026")

TRAIN_JSON = SPLIT_DIR / "train.json"
VAL_JSON = SPLIT_DIR / "val.json"
TEST_JSON = SPLIT_DIR / "test.json"
SPLIT_MANIFEST = SPLIT_DIR / "split_manifest.json"

source_records = json.loads(SOURCE_TRAIN_JSON.read_text(encoding="utf-8"))
train_records = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
val_records = json.loads(VAL_JSON.read_text(encoding="utf-8"))
test_records = json.loads(TEST_JSON.read_text(encoding="utf-8"))
assert len(source_records) == 7000
assert len(train_records) == 5600
assert len(val_records) == 700
assert len(test_records) == 700
train_ids, val_ids, test_ids = set(train_records), set(val_records), set(test_records)
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert train_ids | val_ids | test_ids == set(source_records)

split_manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
assert split_manifest["sampling"]["seed"] == 2026
assert split_manifest["outputs"]["train"]["query_count"] == 5600
assert split_manifest["outputs"]["val"]["query_count"] == 700
assert split_manifest["outputs"]["test"]["query_count"] == 700
assert all(split_manifest["invariants"].values())
print("Split OK:", {"train": 5600, "val": 700, "test": 700})


## Khai báo đường dẫn

Chỉ sửa hai đường dẫn index Dataset bên dưới. Không copy index vào `/kaggle/working`.

In [ ]:
# TODO: thay đúng slug/folder Kaggle Dataset của bạn.
SHORT_INDEX_DIR = Path("/kaggle/input/<short-index-dataset>/vn_embedding_v2_dual_v1")
LONG_INDEX_DIR = Path("/kaggle/input/<long-index-dataset>/vn_embedding_v2_dual_longv1")

CONFIG = RETRIEVAL_ROOT / "configs/vietnamese_embedding_dual_long_rerank.yaml"
OUTPUT_ROOT = Path("/kaggle/working/artifacts/reranker_stage1")
SMOKE_OUTPUT_DIR = OUTPUT_ROOT / "smoke_mining"
MINED_OUTPUT_DIR = OUTPUT_ROOT / "mined_data"

RUN_SMOKE = True
OVERWRITE_FULL_OUTPUT = False


In [ ]:
import faiss
from legal_ir.config import PipelineConfig

required_files = [
    TRAIN_JSON,
    SPLIT_MANIFEST,
    CONFIG,
    SHORT_INDEX_DIR / "manifest.json",
    SHORT_INDEX_DIR / "chunks.jsonl",
    SHORT_INDEX_DIR / "dense.faiss",
    LONG_INDEX_DIR / "manifest.json",
    LONG_INDEX_DIR / "chunks.jsonl",
    LONG_INDEX_DIR / "dense.faiss",
]
for path in required_files:
    assert path.is_file(), f"Thiếu file: {path}"
assert (SHORT_INDEX_DIR / "bm25").is_dir(), "Short index bắt buộc có BM25"

short_manifest = json.loads((SHORT_INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
long_manifest = json.loads((LONG_INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
resolved = PipelineConfig.from_yaml(CONFIG)
expected_model = "AITeamVN/Vietnamese_Embedding_v2"
expected_revision = "18b44161e041bf1d3a333ab5144b5b7b93f914d2"
for name, manifest in [("short", short_manifest), ("long", long_manifest)]:
    dense_manifest = manifest["dense"]
    assert dense_manifest["model_name"] == expected_model, (name, dense_manifest)
    assert dense_manifest["revision"] == expected_revision, (name, dense_manifest)
    assert dense_manifest["max_length"] == 2048, (name, dense_manifest)
    assert dense_manifest["normalize_embeddings"] is True, (name, dense_manifest)
    index = faiss.read_index(str((SHORT_INDEX_DIR if name == "short" else LONG_INDEX_DIR) / "dense.faiss"))
    assert index.ntotal == manifest["chunk_count"], (name, index.ntotal, manifest["chunk_count"])
    print(name, "chunks/vectors:", index.ntotal, "dimension:", index.d)

assert resolved.bm25.top_k_chunks == 50
assert resolved.dense.top_k_chunks == 100
assert resolved.hyde.enabled is False
assert resolved.reranker.enabled is True
assert resolved.reranker.multi_gpu is True
assert resolved.long_context.enabled is True
assert resolved.long_context.candidate_mode == "full"

with (SHORT_INDEX_DIR / "chunks.jsonl").open("r", encoding="utf-8") as handle:
    short_sample = json.loads(handle.readline())
with (LONG_INDEX_DIR / "chunks.jsonl").open("r", encoding="utf-8") as handle:
    long_sample = json.loads(handle.readline())
assert short_sample["metadata"]["granularity"] == "short"
assert short_sample["metadata"]["primary_long_chunk_id"]
assert short_sample["metadata"]["long_chunk_ids"]
assert long_sample["metadata"]["granularity"] == "long"
print("Artifact/config validation OK. Long BM25 không được yêu cầu.")


In [ ]:
def mining_command(output_dir: Path, *, max_queries: int | None = None, overwrite: bool = False):
    command = [
        sys.executable,
        "-m",
        "legal_ir.mine_reranker_stage1",
        "--gold", str(TRAIN_JSON),
        "--split-manifest", str(SPLIT_MANIFEST),
        "--short-index-dir", str(SHORT_INDEX_DIR),
        "--long-index-dir", str(LONG_INDEX_DIR),
        "--config", str(CONFIG),
        "--output-dir", str(output_dir),
        "--positive-dense-top-k", "8",
        "--direct-long-top-k", "60",
        "--negatives-per-positive", "7",
        "--near-margin", "2.0",
        "--dense-query-batch-size", "16",
        "--log-every", "25",
    ]
    if max_queries is not None:
        command.extend(["--max-queries", str(max_queries)])
    if overwrite:
        command.append("--overwrite")
    return command

if RUN_SMOKE:
    subprocess.run(
        mining_command(SMOKE_OUTPUT_DIR, max_queries=5, overwrite=True),
        check=True,
        env=os.environ.copy(),
    )
    smoke_manifest = json.loads((SMOKE_OUTPUT_DIR / "stage1_manifest.json").read_text(encoding="utf-8"))
    assert smoke_manifest["smoke_test"] is True
    assert smoke_manifest["dataset"]["group_count"] > 0
    print("Smoke mining OK:", smoke_manifest["dataset"])


## Mine toàn bộ train split

Đây là cell tốn thời gian. Pretrained reranker chạy data-parallel trên hai GPU. Output được ghi streaming qua file tạm; chỉ publish dataset hoàn chỉnh khi job thành công.

In [ ]:
subprocess.run(
    mining_command(
        MINED_OUTPUT_DIR,
        overwrite=OVERWRITE_FULL_OUTPUT,
    ),
    check=True,
    env=os.environ.copy(),
)


In [ ]:
DATASET_PATH = MINED_OUTPUT_DIR / "stage1_train.jsonl"
DATASET_MANIFEST = MINED_OUTPUT_DIR / "stage1_manifest.json"
SKIPPED_PATH = MINED_OUTPUT_DIR / "stage1_skipped.jsonl"
for path in [DATASET_PATH, DATASET_MANIFEST, SKIPPED_PATH]:
    assert path.is_file(), path

mined_manifest = json.loads(DATASET_MANIFEST.read_text(encoding="utf-8"))
digest = hashlib.sha256()
with DATASET_PATH.open("rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == mined_manifest["dataset"]["sha256"]

with DATASET_PATH.open("r", encoding="utf-8") as handle:
    sample = json.loads(handle.readline())
assert sample["positive"]["document_id"] in sample["gold_document_ids"]
assert len(sample["negatives"]) == 7
assert not ({item["document_id"] for item in sample["negatives"]} & set(sample["gold_document_ids"]))
assert len({item["document_id"] for item in sample["negatives"]}) == 7
print(json.dumps(mined_manifest, ensure_ascii=False, indent=2))
print("Dataset size:", f"{DATASET_PATH.stat().st_size / 1024**3:.3f} GiB")


In [ ]:
# Tùy chọn: bundle dataset nhỏ để tải lên Kaggle Dataset cho notebook training.
from zipfile import ZIP_DEFLATED, ZipFile
from IPython.display import FileLink, display

CREATE_DATASET_ZIP = True
if CREATE_DATASET_ZIP:
    ZIP_PATH = Path("/kaggle/working/reranker_stage1_mined_data.zip")
    with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
        for source in [DATASET_PATH, DATASET_MANIFEST, SKIPPED_PATH]:
            archive.write(source, arcname=f"reranker_stage1_mined_data/{source.name}")
    zip_hash = hashlib.sha256()
    with ZIP_PATH.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            zip_hash.update(block)
    zip_digest = zip_hash.hexdigest()
    checksum = ZIP_PATH.with_suffix(".zip.sha256")
    checksum.write_text(f"{zip_digest}  {ZIP_PATH.name}\n", encoding="utf-8")
    print("Bundle:", ZIP_PATH, f"{ZIP_PATH.stat().st_size / 1024**3:.3f} GiB")
    print("SHA256:", zip_digest)
    display(FileLink(str(ZIP_PATH)))
    display(FileLink(str(checksum)))
